# 🧠 BRISC 2025 Brain Tumor Segmentation & Multi-Task Learning
### Google Colab GPU Accelerated Research Pipeline

This notebook trains and evaluates:
1. **Baseline U-Net** with Monte Carlo Dropout
2. **Multi-Task U-Net** (Tumor Segmentation + 4-Class Pathology Classification)
3. **Attention U-Net** with Attention Gates
4. **Robustness Stress-Testing** across 4 clinical degradation types
5. **Epistemic Uncertainty Quantification**

In [ ]:
# 1. Check GPU Availability
!nvidia-smi

In [ ]:
# 2. Clone Repository or Setup Directory
!git clone https://github.com/your-username/brain-tumor-det-segmentation.git
%cd brain-tumor-det-segmentation
!pip install -r backend/requirements.txt

In [ ]:
# 3. Download BRISC 2025 Dataset from Kaggle
!pip install -q kaggle
# Upload your kaggle.json or set credentials:
# import os
# os.environ['KAGGLE_USERNAME'] = 'your_username'
# os.environ['KAGGLE_KEY'] = 'your_key'
!kaggle datasets download -d briscdataset/brisc2025 -p cv/data --unzip

In [ ]:
# 4. Inspect & Audit Dataset
!python cv/data/inspect_data.py

In [ ]:
# 5. Train Baseline U-Net on GPU (Full Dataset)
from cv.training.train_unet import train_unet

train_unet(
    data_dir='cv/data/brisc2025',
    output_dir='artifacts',
    epochs=25,
    batch_size=16,
    lr=1e-3,
    features=32,
    patience=7
)

In [ ]:
# 6. Train Multi-Task U-Net on GPU
from cv.training.train_multitask import train_multitask

train_multitask(
    data_dir='cv/data/brisc2025',
    output_dir='artifacts',
    epochs=25,
    batch_size=16,
    lambda_cls=0.5,
    features=32,
    patience=7
)

In [ ]:
# 7. Run Held-Out Test Evaluation & Robustness Benchmarking
!python cv/evaluation/evaluate.py
from cv.robustness.benchmark import RobustnessBenchmark
from cv.models.unet import UNet
from cv.data.dataset import BRISCDataset
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = UNet(in_channels=1, out_classes=1, features=32).to(device)
state = torch.load('artifacts/checkpoints/best_unet.pth', map_location=device)
model.load_state_dict(state['model_state_dict'])
model.eval()

test_ds = BRISCDataset('cv/data/brisc2025', split='test')
bench = RobustnessBenchmark(model, device=device)
bench.run_benchmark_suite(test_ds.samples[:100], 'artifacts/metrics')

In [ ]:
# 8. Package Checkpoints & Metrics for Download
!zip -r artifacts.zip artifacts/